# Lesson 08 — Parent-document retrieval and metadata filtering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/08-parent-doc-and-metadata/colab.ipynb)

*Search with small chunks, answer with big ones. And filter *before* you search, not after.*

**What you will learn**

- Small-to-big retrieval: precise matching on small chunks, full context for the generator
- How recorded character offsets turn it into a slice — no second index
- Why overlapping windows must be merged, and the surprising side effect that raised recall
- Pre-filtering vs post-filtering, and the selectivity rule for choosing
- Why a filter that is silently ignored is a permission leak

| | |
|---|---|
| ⏱️ Time | 45 min |
| 🖥️ Runtime | CPU is fine |
| 🤖 Needs the generator (Ollama) | no — retrieval only |

**Before this:** [07-contextual-retrieval](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/07-contextual-retrieval/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = False  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. Small-to-big

Lesson 02 left a tension unresolved:

- **small chunks** retrieve precisely (the match is not diluted) but answer poorly (the
  surrounding context is gone);
- **large chunks** do the reverse.

Small-to-big refuses the trade: **search over small chunks, hand the generator the
larger region each hit came from.**

```
document:  ......................[ small chunk matched ]......................
                         ◄─────── window ──────►[chunk]◄─────── window ───────►
generator sees:          [=============== parent window, snapped =============]
```

In [ ]:
from ragkit.pipeline import RagPipeline

plain = RagPipeline(strategy="rerank")
if plain.count() == 0:
    print(plain.ingest())
documents = {d.doc_id: d for d in plain.documents}

## 2. Where lesson 02's bookkeeping pays off

Every chunk carries `start_char` / `end_char` into its source document. That felt like
fussy overhead when it was introduced. It makes this whole lesson a slice:

```python
parent_text = document.text[start - window : end + window]
```

No second index, no parent/child table, no duplicated storage. Let's do it by hand.

In [ ]:
hit = plain.retrieve("How does chunk overlap work?", top_k=1)[0].chunk
doc = documents[hit.doc_id]
WINDOW = 300

naive_start, naive_end = max(0, hit.start_char - WINDOW), min(len(doc.text), hit.end_char + WINDOW)
print(f"child: {len(hit.text)} chars at {hit.start_char}–{hit.end_char} of {hit.doc_id}\n")
print("naive window starts:", repr(doc.text[naive_start : naive_start + 60]))
print("naive window ends  :", repr(doc.text[naive_end - 60 : naive_end]))

A window cut at an arbitrary character starts mid-word and ends mid-clause — exactly the
damage small-to-big exists to undo. `ragkit` **snaps** each edge to the nearest paragraph
or sentence break:

In [ ]:
from ragkit.retrieve.parent import _snap_to_boundary

start = _snap_to_boundary(doc.text, naive_start, forward=False)
end = _snap_to_boundary(doc.text, naive_end, forward=True)
print("snapped window starts:", repr(doc.text[start : start + 60]))
print("snapped window ends  :", repr(doc.text[end - 60 : end]))

## 3. Child vs parent, side by side

`ParentDocumentRetriever` wraps the reranked retriever and expands each hit. The lesson
script prints the child chunk (what was *matched*) and the parent window (what the
generator *reads*):

In [ ]:
!{sys.executable} lessons/08-parent-doc-and-metadata/run.py "How does chunk overlap work?" --top-k 2

### Overlapping windows are merged, not duplicated

Two chunks 300 characters apart expand into two nearly identical windows. Handing the
generator the same text twice wastes context budget *and* pushes a genuinely different
passage out of the top k. So overlapping windows of one document are merged into a
single result, recorded as `merged_windows`.

In [ ]:
parent = RagPipeline(strategy="parent", parent_window=600)
parent._embedder, parent._reranker = plain.embedder, plain.reranker  # reuse loaded models

question = "What is chunk overlap and what does it cost?"
cols = ["doc_id", "merged_windows", "child_chars", "parent_chars", "rerank", "text"]
frame = results_frame(parent.retrieve(question, top_k=5), chars=60)
frame[[c for c in cols if c in frame.columns]]

## 4. Measure it — and the prediction that was wrong

An earlier draft of this lesson predicted: *"document-level recall should be roughly
unchanged — the same documents are retrieved — while answer quality may improve."*

🔮 **Predict** for yourself: will recall go up, down, or stay flat?

In [ ]:
from ragkit.eval.run import run_eval

runs = {"rerank": run_eval(pipeline=plain, retrieval_only=True, quiet=True)}
for window in (200, 600, 1500):
    p = RagPipeline(strategy="parent", parent_window=window)
    p._embedder, p._reranker = plain.embedder, plain.reranker
    runs[f"parent (window {window})"] = run_eval(pipeline=p, retrieval_only=True, quiet=True)
compare_runs(runs).round(3)

### Two runs, two answers

The repo's original run (37 questions, the reranker's shortlist then at 25) reported:

| Pipeline | R@1 | R@3 | R@5 | nDCG@5 |
|---|---|---|---|---|
| rerank | 0.811 | 0.960 | 0.987 | 0.946 |
| parent | 0.841 | 0.966 | 0.989 | 0.953 |

and explained the +0.030 by **diversity**: merging overlapping windows collapses
same-document hits, freeing top-k slots for other documents.

When this notebook was written (44 questions, current defaults), **parent and rerank
tied exactly on every metric** — the earlier gain did not reproduce. That is worth
understanding rather than shrugging at:

- The harness already collapses each *chunk* ranking into a *document* ranking
  (`dedupe_to_documents`, lesson 03) before scoring. Chunk-level duplicates are
  therefore mostly invisible to document-level recall — diversity can only move these
  numbers in the rare case where the top 20 chunks cover fewer than 5 documents.
- +0.030 at R@1 was **one question** out of 37. A one-question difference under a
  changed configuration is exactly the kind of result lessons 03 and 05 warned about.

So the original prediction — *document-level recall roughly unchanged* — was right
after all. Where the diversity effect is real is in **what the generator reads**:

In [ ]:
import numpy as np

from ragkit.eval.golden import load_golden

questions = [q.question for q in load_golden()]
for name, pipe in [("rerank", plain), ("parent (600)", parent)]:
    distinct = [len({s.chunk.doc_id for s in pipe.retrieve(q, top_k=5)}) for q in questions]
    chars = [sum(len(s.chunk.text) for s in pipe.retrieve(q, top_k=5)) for q in questions]
    print(
        f"{name:14s} distinct docs in top 5: {np.mean(distinct):.2f}   "
        f"context chars handed to the generator: {np.mean(chars):,.0f}"
    )

Here is the real effect: the five passages handed to the generator come from **more
distinct documents** (fewer near-duplicates) and carry **several times more text**. More text costs tokens (cloud) and seconds (local) on
every query, and can bury the answer — measure answer quality, not just recall, before
shipping a big window.

> **The general habit:** when a measurement contradicts your prediction — or a later
> measurement contradicts an earlier one — the mechanism behind the surprise is worth
> more than the number. Here the mechanism was real, but it acts on the prompt, not on
> the metric that was credited with it.

## 5. Metadata filtering — mostly a warning

Real systems restrict retrieval: *only this tenant's documents*, *only documents this
user may see*, *only the 2024 policies*. There are two ways to do it.

| | Post-filter | Pre-filter |
|---|---|---|
| How | retrieve global top k, then drop ineligible results | restrict the candidate set, then search |
| Correct? | **no** — can return almost nothing, silently | yes, by construction |
| Effort | trivial | needs index support (`ChromaStore.search(where=...)`) |

Let's watch a post-filter fail. Restrict to `07-operations.md` (~1/16 of the corpus)
and ask a question that matches *other* documents better:

In [ ]:
from ragkit.retrieve import DenseRetriever
from ragkit.retrieve.rerank import RerankingRetriever

dense = DenseRetriever(plain._store, plain.embedder)
q = "How should configuration settings be tuned for retrieval?"
allowed = "07-operations.md"

global_top = dense.retrieve(q, top_k=5)
post = [s for s in global_top if s.chunk.doc_id == allowed]
pre = dense.retrieve(q, top_k=5, where={"doc_id": allowed})

print("global top 5        :", [s.chunk.doc_id for s in global_top])
print(f"POST-filter results : {len(post)}  ← thin or empty, and nothing raised")
print(f"PRE-filter results  : {len(pre)}  ←", [round(s.score, 3) for s in pre])

The post-filter fails **silently**: a thin result set, no error, and a generator that
now says "the context doesn't cover this" for a question the tenant's own documents
answer.

### The selectivity rule

*Selectivity* = the fraction of the corpus the predicate keeps.

| Selectivity | Approach |
|---|---|
| keeps most of the corpus | post-filter is fine |
| keeps a modest fraction | pre-filter |
| keeps under ~1% | a **separate index per partition** — an approximate index walks a graph of mostly ineligible nodes, and an exact scan over the eligible subset is simpler and faster |

**Whatever you filter on must live on the chunk.** Resolving a permission at query time
puts a network round trip inside retrieval and makes that system's outage your search
outage. Denormalise at ingest.

## 6. The silent-drop trap

What should a retriever do when handed a filter it cannot apply? `Bm25Retriever`
**raises** rather than ignoring it:

In [ ]:
try:
    plain.retriever.retrieve(q, top_k=5, where={"doc_id": allowed})  # hybrid → includes BM25
except NotImplementedError as exc:
    print("NotImplementedError:", exc)

A filter that is silently dropped is a **permission leak** waiting to happen: tenant A's
query returns tenant B's documents and nobody finds out until a customer does. An honest
failure is worth more than a convenient one.

(This is also why the lesson script's `--filter` path filters through a **dense →
rerank** retriever rather than the hybrid pipeline. It originally used the hybrid
pipeline and crashed with exactly this error — found while writing this notebook.)

In [ ]:
filtered = RerankingRetriever(dense, plain.reranker, candidates=10)
results_frame(filtered.retrieve(q, top_k=3, where={"doc_id": allowed}), chars=80)

## 7. Check your understanding

**1. Why does parent-document retrieval need no second index here?**

<details><summary>▶ Show answer</summary>

Because every chunk stores `start_char`/`end_char` into its source document, so the
parent window is a slice of the original text. Without those offsets you would need
a separate parent store and a child→parent mapping.

</details>

**2. Merging windows makes the top 5 more diverse. Why did that barely move document-level recall in this notebook?**

<details><summary>▶ Show answer</summary>

The eval already deduplicates the chunk ranking to a document ranking before
scoring, so duplicate chunks of one document were never costing document-level
recall in the first place. The diversity shows up in what the *generator* reads —
more distinct documents, less repetition — which you would measure with answer
quality, not retrieval recall.

</details>

**3. A tenant owns 0.5% of a 10M-chunk corpus. Which filtering approach?**

<details><summary>▶ Show answer</summary>

A separate index (or partition) per tenant. At that selectivity a post-filter
returns nothing and a pre-filter over an approximate global index wastes most of its
graph walk on ineligible nodes; an exact scan over the tenant's ~50k chunks is
simpler and fast.

</details>

## ✍️ Exercises

1. **Sweep the window** more finely (100 → 3000). Where does recall stop rising, and how
   many context characters does each step cost?
2. **Force a merge.** Find a question where two chunks of the same document both rank in
   the top 5 under `rerank`, then show they merge under `parent`.
3. **Add a metadata field.** Give every chunk a fake `tenant` (e.g. by document number
   parity) at ingest, then compare pre- and post-filtering by tenant over the golden set.

## Key takeaways

1. Search small, answer big: offsets make the parent window a free slice.
2. Snap windows to sentence/paragraph boundaries, and merge overlapping windows.
3. Merging improves the diversity of what the generator reads; a claimed recall gain from it did not reproduce — always re-measure.
4. Pre-filter, not post-filter; for very selective predicates, partition the index.
5. A filter that cannot be applied must fail loudly — silent drops leak data.

## Next

**[Lesson 09 — Routing](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/09-routing-and-multi-index/colab.ipynb)**